# JobRec GNN: Kaggle / Colab end-to-end
Upload repository code and the original JTH CSV files first. Set `JOBREC_ROOT` if the code directory is not the current directory. This notebook does not download or publish private data. Run preparation and inspect coverage before training.

In [ ]:
import os
import sys
from pathlib import Path
candidates = [Path(os.environ.get('JOBREC_ROOT', '.')), Path.cwd(), Path.cwd().parent, Path('/kaggle/working/GNN_for_RCM')]
PROJECT_ROOT = next((path.resolve() for path in candidates if (path / 'src' / 'jobrec').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Set JOBREC_ROOT to the uploaded repository directory before running.')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from jobrec.data import prepare, read_json, write_json
from jobrec.features import fit_preprocessors
from jobrec.eda import write_eda
from jobrec.baselines import evaluate_baseline
print('Project:', PROJECT_ROOT)

## Dependencies
Preparation and basic baselines use only the standard library. For training, install the correct Torch build for this runtime, then run `%pip install torch-geometric`. Install scikit-learn for the Logistic Regression baseline. Do not replace a working Kaggle GPU Torch install blindly. See the README for supported dependency ranges.

In [ ]:
config = read_json(PROJECT_ROOT / 'configs' / 'default.json')
DATA_DIR = Path(os.environ.get('JOBREC_DATA', str(PROJECT_ROOT / 'data')))
if not (DATA_DIR / 'candidates.csv').exists() and Path('/kaggle/input').exists():
    matches = list(Path('/kaggle/input').rglob('candidates.csv'))
    if len(matches) == 1:
        DATA_DIR = matches[0].parent
OUTPUT = Path('/kaggle/working/jobrec-output') if Path('/kaggle/working').exists() else PROJECT_ROOT / 'artifacts' / 'notebook'
config.update(data_dir=str(DATA_DIR), output_dir=str(OUTPUT / 'prepared'))
if not (DATA_DIR / config['jobs_file']).exists() and (DATA_DIR / 'jobs.csv').exists():
    config['jobs_file'] = 'jobs.csv'
print(config['data_dir'], config['jobs_file'], config['output_dir'])

In [ ]:
bundle, audit, coverage = prepare(config)
state = fit_preprocessors(bundle, config)
write_json(OUTPUT / 'prepared' / 'preprocessors.json', state)
write_eda(bundle, audit, OUTPUT / 'prepared' / 'eda')
print(audit['interactions'])
print(coverage)

## Stop and inspect coverage
With the supplied CSVs, the fixed-cutoff protocol retains approximately 6% of held-out pairs after availability checks. This is a restricted cohort, not a whole-dataset performance claim. Read coverage.json and the snapshot caveat before interpreting scores. Do not silently add future profiles or jobs to improve coverage.

In [ ]:
baseline_results = {}
for name in ('random', 'popular', 'content'):
    result, per_user, recommendations = evaluate_baseline(bundle, state, name, 'validation')
    baseline_results[name] = result['metrics']
    write_json(OUTPUT / 'baselines' / name / 'metrics.json', result)
    write_json(OUTPUT / 'baselines' / name / 'per_user.json.gz', per_user)
print(baseline_results)

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    import torch
    from jobrec.training import train, evaluate_checkpoint
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('Device:', device, 'Torch:', torch.__version__)
    for architecture in ('sage', 'rgcn', 'hgt'):
        destination = OUTPUT / architecture
        train(bundle, config, architecture, destination, device=device)
        result, per_user, recommendations = evaluate_checkpoint(bundle, destination / 'best.pt', 'validation', device=device)
        write_json(destination / 'validation_metrics.json', result)
        write_json(destination / 'validation_per_user.json.gz', per_user)
else:
    print('Training disabled. Use configs/smoke.json first, then enable RUN_TRAINING when ready.')

## Next
Select an encoder using validation only. Compare gate modes none/uniform/global/profile/history, then run the multi-seed experiment script. Evaluate test only after the design is frozen. Save checkpoints, config, environment versions and coverage reports. The Streamlit demo is intended for local use; do not publish raw candidate profiles.